In [ ]:
from itertools import islice

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from core.data.augment import Default
from core.training.config import CLASSES, TRAIN_DIR, VIP
from core.data.dataset import MultilabelImageDataset, pos_weight, read_train, sampled_loader


def visualise_dataloader(dl, n_batches=None, with_outputs=True):
    """Per batch: cats and dogs that are not VIP animals, each VIP animal, rows with no cats or dogs; read from labels of drawn rows; images never loaded.
    `n_batches`: only first that many batches, `None` for whole epoch."""
    df = dl.dataset.df
    is_vip = df[VIP].any(axis=1)
    names = ["Cat (no VIP)", "Dog (no VIP)", *VIP, "No cats or dogs"]
    labels = np.column_stack(
        [
            (df["Cat"] == 1) & ~is_vip,
            (df["Dog"] == 1) & ~is_vip,
            df[VIP],
            (df["Cat"] == 0) & (df["Dog"] == 0),
        ]
    ).astype(int)
    idxs_seen = []
    batch_counts = []

    for idxs in islice(dl.batch_sampler, n_batches):
        idxs_seen.extend(idxs)
        batch_counts.append(labels[idxs].sum(axis=0))

    batch_counts = np.array(batch_counts)  # (n_batches, len(names))

    if with_outputs:
        fig, ax = plt.subplots(1, figsize=(15, 5))

        ind = np.arange(len(batch_counts))
        width = 0.8 / len(names)

        for c, name in enumerate(names):
            ax.bar(ind + c * width, batch_counts[:, c], width, label=name)
        if len(batch_counts) == 1:
            ax.xaxis.set_visible(False)
        else:
            ax.set_xlabel("Batch index", fontsize=12)
        ax.set_ylabel("No. of images in batch", fontsize=12)

        plt.legend()
        plt.show()

        for name, n in zip(names, batch_counts.sum(axis=0)):
            print(f"Count of {name}: {n}")
        print("=============")
        print(f"Num. unique images seen: {len(set(idxs_seen))}/{len(dl.dataset)}")
    return batch_counts, idxs_seen

In [ ]:
train_df = read_train()
train_ds = MultilabelImageDataset(train_df, TRAIN_DIR, Default().compose())

In [ ]:
train_dl_raw = sampled_loader(train_ds, 2048, 8, 1, 1)
_ = visualise_dataloader(train_dl_raw, n_batches=1)

In [ ]:
# weights = sample_weights(df, 8, 0.2)
# sampler = torch.utils.data.WeightedRandomSampler(
#     weights,
#     num_samples=round(weights.sum()),
#     replacement=True,
#     generator=torch.Generator().manual_seed(SEED),
# )
# drawn = df.iloc[list(sampler)]

In [ ]:
train_dl_after_magic = sampled_loader(train_ds, 2048, 8, 8, 0.2)
_ = visualise_dataloader(train_dl_after_magic, n_batches=1)
pd.Series(pos_weight(train_df, 8, 0.2, is_new=True).numpy(), index=CLASSES)

In [ ]:
train_dl_no_down = sampled_loader(train_ds, 2048, 8, 8, 1)
_ = visualise_dataloader(train_dl_no_down, n_batches=1)
pd.Series(pos_weight(train_df, 8, 1, is_new=True).numpy(), index=CLASSES)